# Day 32 Tutorial：GCN 与 GraphSAGE 的公平比较

## Goal

在同一 KarateClub 图、固定掩码、隐藏宽度、优化器、120 epoch 和三种配对随机种子下比较 GCN 与 GraphSAGE；报告每次 validation accuracy、mean/std 和参数量。

**边界：** test 标签保持封存；这是一项教学级 validation 比较，不是算法胜负或材料结论。

## Setup

需要 `torch`、`torch_geometric`、`numpy` 与 `pandas`。本 Notebook 不自动安装依赖、不捕获或隐藏训练错误，也不持久化容易受机器状态干扰的墙钟时间。

In [ ]:
import random

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from torch_geometric.datasets import KarateClub
from torch_geometric.nn import GCNConv, SAGEConv

MASK_SEED = 20260728
SEEDS = [7, 17, 27]
HIDDEN_CHANNELS = 16
EPOCHS = 120
DROPOUT = 0.5
LEARNING_RATE = 0.01
WEIGHT_DECAY = 5e-4

print("paired model seeds:", SEEDS)

## Steps

### 1. 一次性建立固定数据与掩码

掩码不随模型或模型种子改变。划分剩余节点时不读取 validation/test 标签。

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def make_fixed_masks(graph, mask_seed=MASK_SEED):
    train_mask = graph.train_mask.clone()
    remaining_nodes = (~train_mask).nonzero(as_tuple=False).view(-1)
    generator = torch.Generator().manual_seed(mask_seed)
    order = torch.randperm(
        remaining_nodes.numel(),
        generator=generator,
    )
    remaining_nodes = remaining_nodes[order]
    valid_count = remaining_nodes.numel() // 2

    valid_mask = torch.zeros(graph.num_nodes, dtype=torch.bool)
    test_mask = torch.zeros(graph.num_nodes, dtype=torch.bool)
    valid_mask[remaining_nodes[:valid_count]] = True
    test_mask[remaining_nodes[valid_count:]] = True
    return train_mask, valid_mask, test_mask


dataset = KarateClub()
data = dataset[0]
train_mask, valid_mask, test_mask = make_fixed_masks(data)

print("x shape:", tuple(data.x.shape))
print("edge_index shape:", tuple(data.edge_index.shape))
print(
    "mask sizes:",
    int(train_mask.sum()),
    int(valid_mask.sum()),
    int(test_mask.sum()),
)

### 2. 定义相同两层外壳的 GCN 与 GraphSAGE

两者的 shape 都是 `[N,F] → [N,16] → [N,C]`，但内部邻居聚合和参数化不同。

In [ ]:
class SmallGCN(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, dropout):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)
        self.dropout = dropout

    def forward(self, x, edge_index):
        hidden = F.relu(self.conv1(x, edge_index))
        hidden = F.dropout(
            hidden,
            p=self.dropout,
            training=self.training,
        )
        return self.conv2(hidden, edge_index)


class SmallGraphSAGE(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, dropout):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, out_channels)
        self.dropout = dropout

    def forward(self, x, edge_index):
        hidden = F.relu(self.conv1(x, edge_index))
        hidden = F.dropout(
            hidden,
            p=self.dropout,
            training=self.training,
        )
        return self.conv2(hidden, edge_index)


def count_trainable_parameters(model):
    return sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )


MODEL_BUILDERS = {
    "GCN": SmallGCN,
    "GraphSAGE": SmallGraphSAGE,
}

for model_name, model_class in MODEL_BUILDERS.items():
    sample_model = model_class(
        dataset.num_features,
        HIDDEN_CHANNELS,
        dataset.num_classes,
        DROPOUT,
    )
    sample_model.eval()
    with torch.no_grad():
        sample_logits = sample_model(data.x, data.edge_index)
    print(
        model_name,
        "logits",
        tuple(sample_logits.shape),
        "parameters",
        count_trainable_parameters(sample_model),
    )

### 3. 用同一个训练函数运行每个模型

每次先固定种子，再创建全新模型和全新优化器。函数只返回 validation accuracy。

In [ ]:
def masked_accuracy(logits, labels, mask):
    predictions = logits.argmax(dim=1)
    return (
        (predictions[mask] == labels[mask])
        .float()
        .mean()
        .item()
    )


def train_one_run(model_class, seed):
    set_seed(seed)
    model = model_class(
        dataset.num_features,
        HIDDEN_CHANNELS,
        dataset.num_classes,
        DROPOUT,
    )
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    for _ in range(EPOCHS):
        model.train()
        optimizer.zero_grad()
        logits = model(data.x, data.edge_index)
        loss = F.cross_entropy(
            logits[train_mask],
            data.y[train_mask],
        )
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        logits = model(data.x, data.edge_index)
        valid_accuracy = masked_accuracy(
            logits,
            data.y,
            valid_mask,
        )
    return model, valid_accuracy, loss.item()

### 4. 三种配对种子全部运行并保留每一行

不挑选最好 seed，不记录墙钟耗时。

In [ ]:
records = []

for seed in SEEDS:
    for model_name, model_class in MODEL_BUILDERS.items():
        trained_model, valid_accuracy, final_train_loss = (
            train_one_run(model_class, seed)
        )
        records.append(
            {
                "model": model_name,
                "seed": seed,
                "valid_accuracy": valid_accuracy,
                "final_train_loss": final_train_loss,
                "parameters": count_trainable_parameters(trained_model),
            }
        )

run_table = pd.DataFrame(records)
display(run_table)

### 5. 汇总 validation mean/std 和参数量

`std` 是三次运行的样本标准差。三次只是教学观察，不能当作统计显著性检验。

In [ ]:
summary = (
    run_table
    .groupby("model", as_index=False)
    .agg(
        valid_mean=("valid_accuracy", "mean"),
        valid_std=("valid_accuracy", "std"),
        parameters=("parameters", "first"),
        runs=("seed", "count"),
    )
)
display(summary)

print(
    "Interpretation: compare mean together with std and parameters; "
    "do not declare a universal winner."
)

## Checks

检查公平协议、结果完整性和掩码边界。test 只参与布尔覆盖检查。

In [ ]:
assert not torch.any(train_mask & valid_mask)
assert not torch.any(train_mask & test_mask)
assert not torch.any(valid_mask & test_mask)
assert torch.all(train_mask | valid_mask | test_mask)

assert len(run_table) == len(SEEDS) * len(MODEL_BUILDERS)
assert set(run_table["seed"]) == set(SEEDS)
assert run_table.groupby("model")["seed"].nunique().eq(len(SEEDS)).all()
assert run_table["valid_accuracy"].between(0, 1).all()
assert np.isfinite(
    run_table[["valid_accuracy", "final_train_loss", "parameters"]]
).all().all()
assert summary["runs"].eq(len(SEEDS)).all()
assert summary["valid_std"].notna().all()

print("Checks passed: paired seeds、完整结果、mask 均正确；test 标签未用于评价。")

## Next Steps

1. 结合参数量和波动写出谨慎结论，不只比较均值的小数点。
2. 完成 `03_exercises.md` 后再看答案。
3. Day 33 原样保留协议，加入隐藏总宽度仍为 16 的 GAT。
4. 继续只用 validation 做开发观察，test 保持封存。